# Archaeological heritage in Greece: official GIS pipeline

This notebook queries public ArcGIS feature layers from Greece’s official Archaeological Cadastre and exports a GeoJSON, CSV, and interactive HTML map. It uses polygon geometries for archaeological sites, historic places, and protection zones, and point geometries for monuments and museums.

**Coverage note:** this is the Ministry of Culture’s official map, but the selected public GIS layers and their current contents should still be checked against the project’s research scope. The service includes mapped extents for several categories; monument and museum layers here use points. Data availability and counts may change.

Source: [Greek Archaeological Cadastre](https://www.arxaiologikoktimatologio.gov.gr/), operated by the Greek Ministry of Culture; data is queried from its public GIS service. Retain source attribution and check the current [terms of use](https://www.arxaiologikoktimatologio.gov.gr/terms-of-use) before redistributing data.


## 1. Setup

Install the packages listed in `requirements.txt` in the notebook's Python environment before running this notebook.

In [1]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode

import folium
import geopandas as gpd
import requests
from requests.adapters import HTTPAdapter
from folium.plugins import MarkerCluster
from shapely.geometry import shape

OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SITE_DIR = Path("site")
SITE_DIR.mkdir(parents=True, exist_ok=True)
ARCGIS_PROXY_URL = "https://ops.arxaiologikoktimatologio.gov.gr/portalapp/proxyportal/proxy.ashx"
ARCGIS_LAYER_ROOT = "https://ops.arxaiologikoktimatologio.gov.gr/gisserverext/rest/services/services_portal/simplegeometries_portal/MapServer"
SOURCE_MAP_URL = "https://www.arxaiologikoktimatologio.gov.gr/map"
REQUEST_TIMEOUT_SECONDS = 120
OBJECT_IDS_PER_REQUEST = 200

DATASETS = [
    {"category": "Archaeological sites", "layer_url": f"{ARCGIS_LAYER_ROOT}/5", "source_layer": "PORTAL_XWROS_Search"},
    {"category": "Archaeological site boundaries (lines)", "layer_url": "https://ops.arxaiologikoktimatologio.gov.gr/gisserverext/rest/services/services_portal/xwros_portal/MapServer/1", "source_layer": "xwros_portal/1"},
    {"category": "Historic places", "layer_url": f"{ARCGIS_LAYER_ROOT}/3", "source_layer": "PORTAL_HISTORICAL_Search"},
    {"category": "Protection zones", "layer_url": f"{ARCGIS_LAYER_ROOT}/7", "source_layer": "PORTAL_ZONES_Search"},
    {"category": "Monuments (points)", "layer_url": f"{ARCGIS_LAYER_ROOT}/0", "source_layer": "PORTAL_MONUMENTS_Search_POI"},
    {"category": "Museums (points)", "layer_url": "https://ops.arxaiologikoktimatologio.gov.gr/gisserverext/rest/services/services_portal/museums_portal/MapServer/0", "source_layer": "museums_portal/0"},
]

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=3))


ModuleNotFoundError: No module named 'geopandas'

## 2. Configure official feature layers

The public map service has separate layers for different geometry types. We download site/history/protection polygons to preserve mapped extents, include the small set of site boundaries mapped as lines, and use point representations for monuments and museums to keep the overview map manageable. Counts are read from the service each time; they are not hard-coded.

In [ ]:
def fetch_arcgis_json(target_url, params):
    query_url = f"{ARCGIS_PROXY_URL}?{target_url}?{urlencode(params)}"
    response = session.get(
        query_url,
        headers={"User-Agent": "GreeceArchaeologyMappingNotebook/1.0 (research project)"},
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    payload = response.json()
    if "error" in payload:
        raise RuntimeError(f"ArcGIS service error: {payload['error']}")
    return payload

retrieved_at_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")
print(f"Using public Greek Archaeological Cadastre GIS layers; snapshot started {retrieved_at_utc}")


## 3. Download and normalize features

The notebook first retrieves each layer’s object IDs, then requests small batches in GeoJSON/WGS84. This avoids silently truncating results at the service’s per-request feature limit. The original service attributes are retained in `source_attributes_json`.

In [ ]:
records = []

def first_value(properties, *keys):
    return next((properties[key] for key in keys if properties.get(key) not in (None, "")), None)

for dataset in DATASETS:
    layer_url = dataset["layer_url"]
    id_payload = fetch_arcgis_json(
        f"{layer_url}/query",
        {"where": "1=1", "returnIdsOnly": "true", "f": "json"},
    )
    object_ids = id_payload.get("objectIds")
    if object_ids is None:
        raise ValueError(f"ArcGIS did not return object IDs for {dataset['category']}: {id_payload}")
    print(f"{dataset['category']}: {len(object_ids):,} records")

    for start in range(0, len(object_ids), OBJECT_IDS_PER_REQUEST):
        batch_ids = object_ids[start : start + OBJECT_IDS_PER_REQUEST]
        batch = fetch_arcgis_json(
            f"{layer_url}/query",
            {
                "objectIds": ",".join(map(str, batch_ids)),
                "outFields": "*",
                "returnGeometry": "true",
                "outSR": "4326",
                "f": "geojson",
            },
        )
        for feature in batch.get("features", []):
            properties = feature.get("properties") or {}
            geometry = feature.get("geometry")
            if geometry is None:
                continue
            records.append({
                "record_key": f"{dataset['source_layer']}/{properties.get('OBJECTID', feature.get('id'))}",
                "source_id": first_value(properties, "ID"),
                "name": first_value(properties, "FIRST_NAME", "NAME"),
                "name_en": first_value(properties, "FIRST_NAME_EN", "NAME_EN"),
                "region_unit": first_value(properties, "FIRST_REGION_UNIT", "REGION_UNIT"),
                "region_unit_en": first_value(properties, "FIRST_REGION_UNIT_EN", "REGION_UNIT_EN"),
                "municipality": first_value(properties, "FIRST_OTA", "OTA"),
                "municipality_en": first_value(properties, "FIRST_OTA_EN", "OTA_EN"),
                "responsible_agency": first_value(properties, "FIRST_FOREAS", "FOREAS"),
                "description": first_value(properties, "FIRST_DESCRIPTION_EN", "DESCRIPTION_EN", "FIRST_DESCRIPTION", "DESCRIPTION"),
                "category": dataset["category"],
                "source_layer": dataset["source_layer"],
                "source_url": SOURCE_MAP_URL,
                "retrieved_at_utc": retrieved_at_utc,
                "source_attributes_json": json.dumps(properties, ensure_ascii=False, sort_keys=True),
                "geometry": shape(geometry),
            })
        time.sleep(0.15)

if not records:
    raise ValueError("The Archaeological Cadastre returned no mappable features.")

sites = gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:4326")
sites = sites.drop_duplicates(subset="record_key").reset_index(drop=True)
sites["longitude"] = sites.geometry.representative_point().x
sites["latitude"] = sites.geometry.representative_point().y
print(f"Prepared {len(sites):,} unique official features")
sites[["record_key", "name", "name_en", "category", "latitude", "longitude"]].head()


## 4. Review the data

Review counts, geometry types, and missing names before treating the results as a complete inventory. Counts describe the current service snapshot.

In [ ]:
display(sites["category"].value_counts(dropna=False).rename_axis("category").to_frame("count"))
display(sites.geometry.geom_type.value_counts().rename_axis("geometry_type").to_frame("count"))
print(f"Rows without a Greek/source name: {sites['name'].isna().sum():,} / {len(sites):,}")


## 5. Export data

GeoJSON preserves point geometry for GIS use; CSV provides a simple tabular export with latitude and longitude.

In [ ]:
geojson_path = OUTPUT_DIR / "greece_archaeological_places.geojson"
csv_path = OUTPUT_DIR / "greece_archaeological_places.csv"

sites.to_file(geojson_path, driver="GeoJSON", index=False)
sites.drop(columns="geometry").to_csv(csv_path, index=False)
print(f"Wrote {geojson_path}")
print(f"Wrote {csv_path}")


## 6. Create an interactive map

Polygon layers retain mapped area boundaries and point layers are clustered. Use the search panel to find places, filter categories, or open a result on the map. The map displays attribution to the Ministry of Culture and OpenStreetMap.

In [ ]:
map_greece = folium.Map(location=[38.0, 23.7], zoom_start=6, tiles="OpenStreetMap")
site_data_dir = SITE_DIR / "data"
site_data_dir.mkdir(parents=True, exist_ok=True)
sidebar = (
    '<button id="menu-button" type="button" aria-label="Open map filters" aria-expanded="false">☰</button>'
    '<div id="sidebar-backdrop" hidden></div>'
    '<aside id="site-sidebar" aria-label="Map search and filters">'
    '<header class="sidebar-header"><p class="eyebrow">Greek Archaeological Cadastre</p>'
    '<h1>Archaeological heritage in Greece</h1>'
    '<p class="sidebar-intro">Explore official archaeological sites, monuments, museums, and protected places.</p>'
    '<button id="close-sidebar" class="sidebar-close" type="button" aria-label="Close map filters">×</button></header>'
    '<div class="sidebar-content"><div class="total-card"><span class="total-label">Mapped places</span>'
    '<span id="total-count">—</span></div><p id="map-load-status" role="status" aria-live="polite"></p>'
    '<section aria-labelledby="search-heading">'
    '<h2 id="search-heading" class="section-heading">Find a place</h2>'
    '<label class="search-box"><span class="search-icon" aria-hidden="true">⌕</span>'
    '<input id="place-search" type="search" placeholder="Search name or region" autocomplete="off" /></label>'
    '<p id="search-status" role="status" aria-live="polite">Type at least 2 characters to search.</p>'
    '<div id="search-results"></div></section><section class="category-section" aria-labelledby="category-heading">'
    '<div class="section-heading"><h2 id="category-heading">Categories</h2>'
    '<div class="category-actions"><button id="show-all" class="text-button" type="button">Show all</button>'
    '<button id="hide-all" class="text-button" type="button">Hide all</button></div></div>'
    '<div id="category-list"></div></section></div>'
    '<footer class="sidebar-footer">Data: <a href="https://www.arxaiologikoktimatologio.gov.gr/" target="_blank" rel="noopener">'
    'Greek Ministry of Culture</a><br>Basemap: OpenStreetMap contributors</footer></aside>'
)
map_greece.get_root().html.add_child(folium.Element(sidebar))
map_greece.get_root().header.add_child(folium.Element('<title>Archaeological Heritage in Greece</title>'))
map_greece.get_root().header.add_child(folium.Element('<link rel="stylesheet" href="style.css">'))
map_greece.get_root().header.add_child(folium.Element('<script defer src="app.js"></script>'))

polygon_types = {"Polygon", "MultiPolygon", "LineString", "MultiLineString"}
map_properties = ["name", "name_en", "category", "region_unit_en", "geometry"]
layer_configs = []
for layer_index, (category, category_sites) in enumerate(sites.groupby("category", sort=True)):
    layer = folium.FeatureGroup(name=category, show=True).add_to(map_greece)
    shapes = category_sites[category_sites.geometry.geom_type.isin(polygon_types)]
    points = category_sites[~category_sites.geometry.geom_type.isin(polygon_types)]
    cluster = MarkerCluster().add_to(layer) if not points.empty else None
    data_files = {"shapes": None, "points": None}

    for kind, features in (("shapes", shapes), ("points", points)):
        if features.empty:
            continue
        data_path = site_data_dir / f"layer_{layer_index}_{kind}.geojson"
        features[map_properties].to_file(data_path, driver="GeoJSON", index=False)
        data_files[kind] = f"data/{data_path.name}"

    layer_configs.append({
        "category": category,
        "variable": layer.get_name(),
        "count": len(category_sites),
        "dataFiles": data_files,
        "clusterVariable": cluster.get_name() if cluster else None,
    })

map_config = {"mapName": map_greece.get_name(), "layers": layer_configs}
config_script = f'<script>window.ARCHAEOLOGY_MAP_CONFIG = {json.dumps(map_config, ensure_ascii=False)};</script>'
map_greece.get_root().html.add_child(folium.Element(config_script))
folium.LayerControl().add_to(map_greece)
map_path = SITE_DIR / "greece_archaeological_places_map.html"
map_greece.save(map_path)
print(f"Wrote {map_path} and map data to {site_data_dir}")
map_greece


## Next steps

- Use the category filter to show archaeological sites, historic places, protection zones, monuments, and museums separately.
- Review source attributes, dates, and geometry accuracy before analysis.
- The source's public layer counts can change; compare the exported counts to those shown by the portal.
- Check the official terms before redistributing source data or derivative datasets.
